# Churn Prediction — Labeled (Subscription E-Commerce)

This notebook trains a churn model on the **Labeled** dataset only. Run **00_data_preprocessing** and **01_eda** first.

**Churn definition (Labeled):** Customer **cancelled or paused** their subscription (explicit label from the dataset).

**Model choice:** We use **Logistic Regression** with `class_weight='balanced'` — the dataset has ~2k rows, so a simpler, interpretable model reduces overfitting risk and gives coefficient interpretation for business use. Scaling is applied for Logistic Regression. Alternative: **Random Forest** with limited depth if you need non-linear patterns.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
DATA_PATH = Path("../data/processed")
OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Labeled processed data

In [ ]:
FEATURES_CSV = DATA_PATH / "labeled_churn_features.csv"
if not FEATURES_CSV.exists():
    raise FileNotFoundError(
        f"Processed data not found at {FEATURES_CSV}. "
        "Run 00_data_preprocessing.ipynb first."
    )
df = pd.read_csv(FEATURES_CSV)
print("Shape:", df.shape)
df.head()

## 2. Prepare features and target

Features: RFM + tenure only (Labeled dataset has no **avg_review**).

In [ ]:
feature_cols = ["recency", "tenure_days", "frequency", "monetary"]
model_df = df.dropna(subset=feature_cols + ["churn"])
X = model_df[feature_cols]
y = model_df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape[0], "Test:", X_test.shape[0], "Churn rate:", y.mean().round(3))

## 3. Train model (Logistic Regression)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

## 4. Evaluation

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_prob)

metrics = {"precision": precision, "recall": recall, "f1": f1, "roc_auc": auc}
for k, v in metrics.items():
    print(f"{k}: {v:.3f}")
print("\nF1 >= 0.70:", "Yes" if metrics["f1"] >= 0.70 else "No (tune threshold or model)")

with open(OUT_DIR / "labeled_churn_model_metrics.json", "w") as f:
    json.dump({k: round(v, 4) for k, v in metrics.items()}, f, indent=2)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
plt.title("Labeled — Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 5. Churn probabilities → high-risk customers (Labeled)

In [ ]:
X_full_scaled = scaler.transform(model_df[feature_cols])
all_prob = model.predict_proba(X_full_scaled)[:, 1]
risk_df = model_df[["customer_id"]].copy()
risk_df["churn_probability"] = all_prob
risk_df = risk_df.sort_values("churn_probability", ascending=False).reset_index(drop=True)

high_risk_path = OUT_DIR / "labeled_high_risk_customers.csv"
risk_df.to_csv(high_risk_path, index=False)
print(f"Saved {len(risk_df)} rows to {high_risk_path}")
risk_df.head(10)